# Assignment 2: RAG-Enhanced Pricing Agent with Historical Knowledge

## Objective
Add factual grounding to our pricing agent using a **pricing knowledge base** with historical data, category benchmarks, and proven pricing strategies.

## Requirements
**RAG Knowledge Base Includes:**
- Category-level elasticity benchmarks
- Historical margins
- 20-50 sample past pricing decisions
- Competitor trends
- Price recommendation guidelines

**Agent Behavior:**
- Pulls past examples
- Justifies price using retrieved data
- Reduces hallucination compared to Iteration 1

## Setup & Dependencies

Install the required packages for RAG implementation.

In [1]:
# Install required packages for RAG
!pip install -q langchain langchain-groq langchain-community
!pip install -q chromadb sentence-transformers
!pip install -q pandas numpy

In [4]:
# Import required libraries
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_text_splitters import CharacterTextSplitter
import os
import getpass
import pandas as pd
import json
from typing import List, Dict


In [5]:
# Set up your Groq API key
print("Please enter your Groq API key:")
print("(You can get one free at: https://console.groq.com/)")
groq_api_key = getpass.getpass("Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key
print("API key set successfully!")

Please enter your Groq API key:
(You can get one free at: https://console.groq.com/)
API key set successfully!


In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

## Create Pricing Knowledge Base

Create a comprehensive pricing knowledge base with historical data and benchmarks.

In [25]:
# TODO: Create pricing knowledge base data structures
# Create the following data structures:

# 1. Category-level elasticity benchmarks
# HINT: elasticity_benchmarks = [ {"category": "Electronics", "elasticity": "High", ...} ]
elasticity_benchmarks = [
    {"category": "Electronics", "elasticity": "High", "description": "Highly sensitive to price changes, small price drops can lead to large increases in demand."},
    {"category": "Clothing", "elasticity": "Medium", "description": "Moderately sensitive to price changes, consumers may switch brands but still purchase within the category."},
    {"category": "Groceries", "elasticity": "Low", "description": "Less sensitive to price changes, consumers need these products regardless of price fluctuations."},
    {"category": "Furniture", "elasticity": "Medium-High", "description": "Sensitive to price changes, especially for non-essential items, but may have some brand loyalty."}
]

# 2. Historical margin data by category  
# HINT: historical_margins = [ {"category": "Electronics", "avg_margin": "12-18%", ...} ]
historical_margins = [
    {"category": "Electronics", "avg_margin": "12-18%", "description": "Margins can vary widely based on product type, with high-end electronics often commanding higher margins."},
    {"category": "Clothing", "avg_margin": "30-50%", "description": "Fashion items typically have higher margins, especially for premium brands."},
    {"category": "Groceries", "avg_margin": "5-15%", "description": "Margins are generally lower due to high competition and price sensitivity."},
    {"category": "Furniture", "avg_margin": "20-40%", "description": "Margins can be substantial, particularly for custom or high-end pieces."}
]

# 3. Sample past pricing decisions (20-50 examples)
# HINT: Include product, category, cost, recommended_price, margin, competitor_price, outcome, reasoning
pricing_decisions = [
    # TODO: Add at least 7 pricing decision examples
    {
        "product": "Samsung Galaxy Smartphone",
        "category": "Electronics",
        "cost": 450,
        "recommended_price": 499,
        "margin": "11%",
        "competitor_price": 529,
        "outcome": "Successful - gained 15% market share",
        "reasoning": "Aggressive pricing in high elasticity category drove volume"
    },
    {
        "product": "Levi's Jeans",
        "category": "Clothing",
        "cost": 30,
        "recommended_price": 49.99,
        "margin": "66%",
        "competitor_price": 59.99,
        "outcome": "Successful - maintained market share",
        "reasoning": "Competitive pricing in medium elasticity category retained customers"
    },
    {
        "product": "Organic Milk",
        "category": "Groceries",
        "cost": 3,
        "recommended_price": 3.49,
        "margin": "16%",
        "competitor_price": 3.99,
        "outcome": "Successful - increased sales by 10%",
        "reasoning": "Slightly lower price in low elasticity category attracted price-sensitive customers"
    },
    {
        "product": "IKEA Dining Table",
        "category": "Furniture",
        "cost": 150,
        "recommended_price": 199,
        "margin": "33%",
        "competitor_price": 249,
        "outcome": "Successful - gained market share",
        "reasoning": "Aggressive pricing in medium-high elasticity category drove volume"
    },
    {
        "product": "Apple MacBook Pro",
        "category": "Electronics",
        "cost": 1200,
        "recommended_price": 1299,      
        "margin": "8%",
        "competitor_price": 1399,
        "outcome": "Successful - maintained market share",
        "reasoning": "Premium pricing in high elasticity category maintained brand perception"
    },
    {
        "product": "Nike Running Shoes",
        "category": "Clothing",         
        "cost": 60,
        "recommended_price": 89.99,             
        "margin": "50%",
        "competitor_price": 99.99,
        "outcome": "Successful - increased sales by 20%",
        "reasoning": "Competitive pricing in medium elasticity category attracted customers from competitors"
    },
    {
        "product": "Whole Wheat Bread",         
        "category": "Groceries",
        "cost": 2,                  
        "recommended_price": 2.49,
        "margin": "25%",
        "competitor_price": 2.99,
        "outcome": "Successful - increased sales by 15%",
        "reasoning": "Slightly lower price in low elasticity category attracted price-sensitive customers"
    }   
]

# 4. Price recommendation guidelines
# HINT: Include rules for high elasticity, competitive positioning, etc.
pricing_guidelines = [
    "For high elasticity categories, consider aggressive pricing strategies to drive volume and gain market share.",
    "For medium elasticity categories, maintain competitive pricing to retain customers while ensuring healthy margins.",
    "For low elasticity categories, focus on value proposition and brand loyalty rather than price competition.",
    "Always analyze competitor pricing and market trends before making pricing decisions.",
    "Consider the cost structure and desired margin when setting prices, but also factor in customer perception and willingness to pay."
]

print(f"Created knowledge base with:")
print(f"- {len(elasticity_benchmarks)} category elasticity benchmarks")
print(f"- {len(historical_margins)} historical margin references") 
print(f"- {len(pricing_decisions)} past pricing decisions")
print(f"- {len(pricing_guidelines)} pricing guidelines")

Created knowledge base with:
- 4 category elasticity benchmarks
- 4 historical margin references
- 7 past pricing decisions
- 5 pricing guidelines


## Create Vector Store for RAG

Convert our knowledge base into searchable documents and create embeddings.

In [26]:
def create_pricing_documents() -> List[Document]:
    """Convert pricing knowledge into searchable documents"""
    documents = []
    
    # TODO: Add elasticity benchmarks to documents
    # HINT: Create Document objects with page_content and metadata
    for benchmark in elasticity_benchmarks:
        content = f"Category: {benchmark['category']}\nElasticity: {benchmark['elasticity']}\nDescription: {benchmark['description']}"
        metadata = {"type": "elasticity_benchmark", "category": benchmark["category"]}
        documents.append(Document(page_content=content, metadata=metadata))
    
    # TODO: Add historical margins to documents
    for margin in historical_margins:
        content = f"Category: {margin['category']}\nAverage Margin: {margin['avg_margin']}\nDescription: {margin['description']}"
        metadata = {"type": "historical_margin", "category": margin["category"]}
        documents.append(Document(page_content=content, metadata=metadata))
    
    # TODO: Add pricing decisions to documents
    for decision in pricing_decisions:
        content = f"Product: {decision['product']}\nCategory: {decision['category']}\nCost: {decision['cost']}\nRecommended Price: {decision['recommended_price']}\nMargin: {decision['margin']}\nCompetitor Price: {decision['competitor_price']}\nOutcome: {decision['outcome']}\nReasoning: {decision['reasoning']}"
        metadata = {"type": "pricing_decision", "category": decision["category"], "product": decision["product"]}
        documents.append(Document(page_content=content, metadata=metadata)) 
    
    # TODO: Add pricing guidelines to documents
    for guideline in pricing_guidelines:
        content = f"Guideline: {guideline}"
        metadata = {"type": "pricing_guideline"}
        documents.append(Document(page_content=content, metadata=metadata))
    
    return documents

# Create documents
pricing_docs = create_pricing_documents()
print(f"Created {len(pricing_docs)} searchable documents")

Created 20 searchable documents


In [ ]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Creating vector store...")

db_path = "./db_home"

vectorstore = Chroma.from_documents(
    pricing_docs,
    embeddings,
    persist_directory=db_path
)

print("Vector store created successfully!")
print(f"Indexed {len(pricing_docs)} documents for retrieval")


Setting up embeddings...
Creating vector store...


Setting up embeddings...
Creating vector store...


InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

## RAG-Enhanced Pricing Agent

Create our enhanced pricing agent that uses retrieval to ground its recommendations.

In [29]:
class RAGPricingAgent:
    def __init__(self, vectorstore, embeddings):
        self.llm = llm
        
        # Store the vector store and embeddings
        self.vectorstore = vectorstore
        self.embeddings = embeddings
        
        self.system_message = SystemMessage(content="""
        You are a RAG-enhanced pricing agent that provides data-driven price recommendations for retail products.
        You have access to a knowledge base of historical pricing decisions, category benchmarks, and pricing guidelines that you can retrieve relevant information from to inform your recommendations.
        """)
        
    def retrieve_relevant_knowledge(self, query: str, k: int = 5):
        """Retrieve relevant pricing knowledge for the query"""
        # TODO: Implement retrieval logic

        # HINT: Use self.vectorstore.as_retriever()
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})
        relevant_docs = retriever.invoke(query)
        return relevant_docs
    
    def format_retrieved_context(self, docs):
        """Format retrieved documents into context string"""
        # TODO: Format documents into a readable context string
        # HINT: Include document content and metadata type
        context_parts = []
        for i, doc in enumerate(docs, 1):
            context_parts.append(f"Document {i} (Type: {doc.metadata['type']}):\n{doc.page_content}")
        
        return "\n\n".join(context_parts)
    
    def get_rag_price_recommendation(self, product_name, category, cost_price, 
                                   current_price=None, target_margin=None, 
                                   competitor_price=None, price_elasticity=None):
        """Get price recommendation using RAG"""
        
        # TODO: Create search query from inputs
        search_query = f"Product: {product_name}\nCategory: {category}\nCost Price: {cost_price}\nCurrent Price: {current_price}\nTarget Margin: {target_margin}\nCompetitor Price: {competitor_price}\nPrice Elasticity: {price_elasticity}"  
            
        # TODO: Retrieve relevant knowledge
        relevant_docs = self.retrieve_relevant_knowledge(search_query)
        context = self.format_retrieved_context(relevant_docs)
        
        # TODO: Create the prompt with retrieved context
        prompt = f"""
You are a pricing agent tasked with recommending an optimal price for the following product:
- Product Name: {product_name}
- Category: {category}   
- Cost Price: {cost_price}
- Current Price: {current_price}
- Target Margin: {target_margin}
- Competitor Price: {competitor_price}
- Price Elasticity: {price_elasticity}
- Retrieved Context: {context}
Provide a price recommendation based on the product details and the retrieved knowledge. Justify your recommendation with reference to the relevant documents and pricing principles.
        """
        
        # Get response from LLM
        messages = [self.system_message, HumanMessage(content=prompt)]
        response = self.llm.invoke(messages)
        
        return response.content

# Initialize the RAG pricing agent
print("Initializing RAG Pricing Agent...")
# rag_agent = RAGPricingAgent(vectorstore, embeddings)
print("RAG Pricing Agent Ready!")

Initializing RAG Pricing Agent...
RAG Pricing Agent Ready!


## Testing the RAG-Enhanced Agent

Test our RAG agent with the assignment example.

In [30]:
rag_agent = RAGPricingAgent(vectorstore, embeddings)

# Test with the assignment example: Puma sneakers
print("Testing Assignment Example: Puma Sneakers")
print("=" * 60)

result = rag_agent.get_rag_price_recommendation(
    product_name="Puma Sneakers",
    category="Footwear", 
    cost_price=1800,  # Note: High cost from assignment
    target_margin=30,
    price_elasticity="High"
)

print(result)

Testing Assignment Example: Puma Sneakers
To recommend an optimal price for the Puma Sneakers, I will analyze the provided product details and the retrieved knowledge from the documents.

The product details are as follows:
- Product Name: Puma Sneakers
- Category: Footwear
- Cost Price: 1800
- Current Price: None
- Target Margin: 30
- Competitor Price: None
- Price Elasticity: High

Given the high price elasticity of the product, it means that customers are highly sensitive to price changes. This implies that a small increase in price may lead to a significant decrease in demand.

The target margin is 30%, which means the selling price should be 1800 / (1 - 0.3) = 2571.43. However, considering the high price elasticity, we should be cautious not to overprice the product.

Looking at the retrieved documents, Document 5 is relevant to our case, as it deals with a high elasticity category (Electronics). The document shows that a premium pricing strategy (8% margin) was used for the Apple

In [31]:
# TODO: Compare RAG vs Baseline approaches
# Create a function that gets both RAG and prompt-only recommendations
# and shows the difference

rag_agent = RAGPricingAgent(vectorstore, embeddings)

def compare_rag_vs_baseline(product_name, category, cost_price, target_margin=None, 
                           competitor_price=None, price_elasticity=None):
    """Compare RAG recommendation with baseline prompt-only approach"""
    
    # TODO: Get RAG recommendation
    rag_result = rag_agent.get_rag_price_recommendation(
        product_name=product_name,
        category=category,
        cost_price=cost_price,          
        target_margin=target_margin,
        competitor_price=competitor_price,
        price_elasticity=price_elasticity
    )
    
    # TODO: Get baseline recommendation (prompt-only)
    baseline_prompt = f"""You are a pricing agent that provides price recommendations for retail products based on product details and general pricing principles. 
    Product Name: {product_name}
    Category: {category}
    Cost Price: {cost_price}
    Target Margin: {target_margin}
    Competitor Price: {competitor_price}  
    Price Elasticity: {price_elasticity}
    Provide a price recommendation based on the product details and general pricing principles, without access to any external knowledge or historical data.      
    """
    baseline_response = llm.invoke([SystemMessage(content="You are a pricing agent that provides price recommendations based on product details."), HumanMessage(content=baseline_prompt)]  )
    
    # TODO: Format and return comparison
    return f"""
    RAG Recommendation:
    {rag_result}    

    Baseline Recommendation:
    {baseline_response.content}
    """

# Test comparison
print("COMPARISON: RAG vs Baseline Approaches")
print("=" * 70)

comparison = compare_rag_vs_baseline(
    "Puma Sneakers",
    "Footwear",
    1800,
    target_margin=30,
    price_elasticity="High"
)

print(comparison)

COMPARISON: RAG vs Baseline Approaches

    RAG Recommendation:
    To recommend an optimal price for the Puma Sneakers, I will analyze the provided product details and the retrieved knowledge from the documents.

The product details are as follows:
- Product Name: Puma Sneakers
- Category: Footwear
- Cost Price: 1800
- Current Price: None
- Target Margin: 30
- Competitor Price: None
- Price Elasticity: High

Given the high price elasticity of the product, it means that customers are highly sensitive to price changes. This implies that a small increase in price may lead to a significant decrease in demand.

The target margin is 30%, which means the selling price should be 1800 / (1 - 0.3) = 2571.43. However, considering the high price elasticity, we should be cautious not to overprice the product.

Looking at the retrieved documents, Document 5 is relevant to our case, as it deals with a high elasticity category (Electronics). The document shows that a premium pricing strategy (8% marg

## Additional Test Cases

Test with various scenarios to see how RAG improves recommendations.

In [37]:
print("Test Case 1: Electronics Category")
print("=" * 40)

electronics_result = rag_agent.get_rag_price_recommendation(
    product_name="Sony 4K TV",
    category="Electronics",
    cost_price=600,
    target_margin=20,
    competitor_price=899,
    price_elasticity="High"
)

print(electronics_result)


Test Case 1: Electronics Category
Based on the product details and the retrieved knowledge, I recommend a price of $839 for the Sony 4K TV.

Justification:

1. **Target Margin**: The target margin for the Sony 4K TV is 20%. To achieve this, we need to calculate the selling price based on the cost price. The cost price is $600, and a 20% margin would result in a selling price of $600 + (20% of $600) = $600 + $120 = $720.
2. **Competitor Price**: The competitor price is $899, which is higher than our calculated selling price of $720. This suggests that we have room to increase our price while still being competitive.
3. **Price Elasticity**: The price elasticity for the Sony 4K TV is high, which means that customers are sensitive to price changes. In high elasticity categories, aggressive pricing can drive volume (Document 1 and Document 2). However, since we are not trying to gain market share aggressively, we can aim for a price that balances competitiveness with profitability.
4. **Ca

In [38]:
# TODO: Test Case 2: Luxury Goods (Low Elasticity)
print("Test Case 2: Luxury Goods Category")
print("=" * 40)

luxury_result = rag_agent.get_rag_price_recommendation(
    product_name="Luxury Watch",
    category="Luxury Goods",
    cost_price=2000,
    target_margin=30,
    competitor_price=3500,
    price_elasticity="Low"
)

print(luxury_result)

Test Case 2: Luxury Goods Category
To determine the optimal price for the Luxury Watch, we need to consider the provided product details and the insights gained from the retrieved documents.

1. **Category and Price Elasticity**: The Luxury Watch falls under the Luxury Goods category, and it has a low price elasticity. This means that changes in price will have a relatively small effect on demand. This characteristic is somewhat similar to the high-end product mentioned in Document 5 (Apple MacBook Pro), where premium pricing maintained brand perception despite high elasticity. However, given the low elasticity of the Luxury Watch, we can infer that customers are less sensitive to price changes, allowing for potentially higher pricing without significantly impacting demand.

2. **Target Margin**: The target margin for the Luxury Watch is 30%. This means the selling price should be such that the profit margin, calculated as (Selling Price - Cost Price) / Selling Price, equals 30%. Using

## Exploring the Knowledge Base

See what the RAG system retrieves for different queries.

In [33]:
def explore_knowledge_retrieval(query, k=3):
    """Show what gets retrieved for a given query"""
    print(f"Query: '{query}'")
    print("=" * 50)
    
    # TODO: Implement retrieval exploration
    # Get documents for the query and display them
    docs = rag_agent.retrieve_relevant_knowledge(query, k=k)
    
    for i, doc in enumerate(docs, 1):
        print(f"Document {i} (Type: {doc.metadata['type']}):\n{doc.page_content}\n{'-' * 50}")
    
    return docs

# Test different retrieval queries
explore_knowledge_retrieval("high elasticity pricing strategy")
print("\n" + "=" * 70 + "\n")
explore_knowledge_retrieval("footwear sneakers margin")

Query: 'high elasticity pricing strategy'
Document 1 (Type: pricing_guideline):
Guideline: For high elasticity categories, consider aggressive pricing strategies to drive volume and gain market share.
--------------------------------------------------
Document 2 (Type: pricing_guideline):
Guideline: For high elasticity categories, consider aggressive pricing strategies to drive volume and gain market share.
--------------------------------------------------
Document 3 (Type: pricing_guideline):
Guideline: For medium elasticity categories, maintain competitive pricing to retain customers while ensuring healthy margins.
--------------------------------------------------


Query: 'footwear sneakers margin'
Document 1 (Type: historical_margin):
Category: Clothing
Average Margin: 30-50%
Description: Fashion items typically have higher margins, especially for premium brands.
--------------------------------------------------
Document 2 (Type: historical_margin):
Category: Clothing
Average Ma

[Document(metadata={'type': 'historical_margin', 'category': 'Clothing'}, page_content='Category: Clothing\nAverage Margin: 30-50%\nDescription: Fashion items typically have higher margins, especially for premium brands.'),
 Document(metadata={'category': 'Clothing', 'type': 'historical_margin'}, page_content='Category: Clothing\nAverage Margin: 30-50%\nDescription: Fashion items typically have higher margins, especially for premium brands.'),
 Document(metadata={'type': 'pricing_decision', 'category': 'Clothing', 'product': 'Nike Running Shoes'}, page_content='Product: Nike Running Shoes\nCategory: Clothing\nCost: 60\nRecommended Price: 89.99\nMargin: 50%\nCompetitor Price: 99.99\nOutcome: Successful - increased sales by 20%\nReasoning: Competitive pricing in medium elasticity category attracted customers from competitors')]

## Your Turn: Expand the Knowledge Base

**Exercise 1:** Add 5 more pricing decisions to the knowledge base
**Exercise 2:** Add a new category with appropriate benchmarks
**Exercise 3:** Test how the new data affects recommendations

In [ ]:
# Exercise 1: Add your pricing decisions here
additional_pricing_decisions = [
    # TODO: Add 5 more pricing decision examples
    # Follow the format from the original pricing_decisions list
]

# Exercise 2: Add a new category
new_category_data = {
    "elasticity_benchmark": {
        # TODO: Add elasticity data for your new category
    },
    "historical_margins": {
        # TODO: Add margin data for your new category
    }
}

# Exercise 3: Test with new category
# TODO: Test your new category with the agent

print("Exercises completed! Test your enhancements.")